### Import Library & Load Data

In [21]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from xgboost import XGBRegressor
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    KFold,
    cross_val_predict
)
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error,
)

pd.set_option('display.float_format', lambda x: '%.3f' % x)


data_prep = joblib.load('../dataset/data_hasil_preprocessing.pkl')

# Mengambil Data Training dan Testing
X_train = data_prep['X_train']
X_test = data_prep['X_test']
y_train = data_prep['y_train']
y_test = data_prep['y_test']

# Mengambil Scaler dan Label Encoder
scaler = data_prep['scaler']
le_district = data_prep['le_district']
le_city = data_prep['le_city']

print("Data hasil preprocessing berhasil dimuat!")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

Data hasil preprocessing berhasil dimuat!
X_train : (6053, 7)
X_test  : (1514, 7)
y_train : (6053,)
y_test  : (1514,)


### Fungsi Perhitungan Metrik Evaluasi

In [23]:
def hitung_metrik_jurnal(y_aktual, y_prediksi, jumlah_fitur):
    n = len(y_aktual)
    r2   = r2_score(y_aktual, y_prediksi)
    mae  = mean_absolute_error(y_aktual, y_prediksi)
    mse  = mean_squared_error(y_aktual, y_prediksi)
    rmse = np.sqrt(mse)
   
    adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - jumlah_fitur - 1))
    
    return {
        'R2': round(r2, 3),
        'Mean absolute error': round(mae, 3),
        'Mean squared error': round(mse, 3),
        'Root mean squared error': round(rmse, 3),
        'Adjusted R2': round(adj_r2, 3)
    }

print("Fungsi perhitungan 6 metrik evaluasi berhasil dibuat!")

Fungsi perhitungan 6 metrik evaluasi berhasil dibuat!


### Hypertuning Parameter

In [24]:
# Inisialisasi Model Dasar XGBoost
xgb_base = XGBRegressor(
    random_state=42,
    n_jobs=1
)

# Skema 10-Fold Cross-Validation
kf = KFold(
    n_splits=10, 
    shuffle=True, 
    random_state=42
)

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [1.0, 1.5, 2.0]
}

# # Hyperparameter Tuning XGBoost dengan GridSearchCV

# grid_search = GridSearchCV(
#     estimator=xgb_base,
#     param_grid=param_grid,
#     cv=kf,
#     scoring='neg_root_mean_squared_error',
#     n_jobs=-1,
#     verbose=1
# )

# print("Memulai GridSearchCV...")
# grid_search.fit(X_train, y_train)

# # Menampilkan Hasil Terbaik
# best_params = grid_search.best_params_

# print("\nHYPERPARAMETER TERBAIK: ")
# for param, nilai in best_params.items():
#     print(f"{param} : {nilai}")

# print(f"\nBest RMSE: {-grid_search.best_score_:.3f}")

# # Hyperparameter Tuning XGBoost dengan RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=1000,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Memulai RandomizedSearchCV...")
random_search.fit(X_train, y_train)

# Menampilkan Hasil Terbaik
best_params = random_search.best_params_

print("\nHYPERPARAMETER TERBAIK:")
for param, nilai in best_params.items():
    print(f"{param} : {nilai}")

print(f"\nBest RMSE: {-random_search.best_score_:.3f}")



Memulai RandomizedSearchCV...
Fitting 10 folds for each of 1000 candidates, totalling 10000 fits

HYPERPARAMETER TERBAIK:
subsample : 0.8
reg_lambda : 1.0
reg_alpha : 0.1
n_estimators : 200
min_child_weight : 3
max_depth : 7
learning_rate : 0.1
gamma : 0
colsample_bytree : 0.8

Best RMSE: 0.431


### Pelatihan dan Pengujian Model XGBoost Final

In [25]:
# Inisialisasi model XGBoost final menggunakan parameter terbaik
model_xgb = XGBRegressor(
    **best_params,
    random_state=42,
    n_jobs=1
)

# Validasi Cross Validation pada data training
kf = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

y_pred_train_xgb = cross_val_predict(
    model_xgb,
    X_train,
    y_train,
    cv=kf
)

# Melatih model pada seluruh data training
model_xgb.fit(X_train, y_train)

# Melakukan prediksi pada data testing
y_pred_test_xgb = model_xgb.predict(X_test)

# Evaluasi model XGBoost
metrik_train_xgb = hitung_metrik_jurnal(
    y_train,
    y_pred_train_xgb,
    jumlah_fitur=X_train.shape[1]
)

metrik_test_xgb = hitung_metrik_jurnal(
    y_test,
    y_pred_test_xgb,
    jumlah_fitur=X_train.shape[1]
)

# Membuat tabel evaluasi
df_tabel_xgb = pd.DataFrame({
    f'Training ({len(X_train)})': metrik_train_xgb,
    f'Testing ({len(X_test)})': metrik_test_xgb
})

print("HASIL PREDIKSI MENGGUNAKAN XGBOOST REGRESSION (TUNED FINAL):")
display(df_tabel_xgb)

HASIL PREDIKSI MENGGUNAKAN XGBOOST REGRESSION (TUNED FINAL):


,Training (6053),Testing (1514)
R2,0.812,0.794
Mean absolute error,0.264,0.285
Mean squared error,0.186,0.213
Root mean squared error,0.432,0.462
Adjusted R2,0.812,0.793


### Menyimpan Hasil XGBoost

In [27]:
# Simpan Model XGBoost 
joblib.dump(model_xgb, '../output/house_price_xgb_model.pkl')
print("✅ Model berhasil disimpan: output/house_price_xgb_model.pkl")

# Simpan Scaler 
joblib.dump(scaler, '../output/scaler.pkl')
print("✅ Scaler berhasil disimpan: output/scaler.pkl")

# Simpan Label Encoder 
joblib.dump({
    'le_district': le_district,
    'le_city': le_city
}, '../output/encoders.pkl')
print("✅ Label Encoders berhasil disimpan: output/encoders.pkl")

# Simpan Hasil Evaluasi 
hasil_xgb = {
    'Model': 'XGBoost',
    'Best_Params': best_params,
    'Training': metrik_train_xgb,
    'Testing': metrik_test_xgb,
    'y_test': y_test,
    'y_pred_test': y_pred_test_xgb
}

joblib.dump(hasil_xgb, '../output/hasil_xgb.pkl')
print("✅ Hasil evaluasi XGBoost berhasil disimpan: output/hasil_xgb.pkl")

✅ Model berhasil disimpan: output/house_price_xgb_model.pkl
✅ Scaler berhasil disimpan: output/scaler.pkl
✅ Label Encoders berhasil disimpan: output/encoders.pkl
✅ Hasil evaluasi XGBoost berhasil disimpan: output/hasil_xgb.pkl
